In [21]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



In [ ]:

from flask import Flask

app=Flask(__name__)

@app.route("/")
def home():
    return "Hello World!"

if __name__=="__main__":
    app.run(host="0.0.0.0",port=3000)


In [27]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [29]:
df.shape

(45211, 17)

In [68]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')

In [72]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,0


In [150]:
df.groupby('y').default.value_counts()

y  default
0  no         39159
   yes          763
1  no          5237
   yes           52
Name: count, dtype: int64

In [31]:
# ---- Step 1: Inspect the data ----
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nTarget distribution:\n", df['y'].value_counts())

# ---- Step 2: Convert target variable 'y' to binary ----
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# ---- Step 3: Identify categorical and numerical columns ----
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('y')

print("\nCategorical features:", categorical_cols)
print("Numerical features:", numerical_cols)

# ---- Step 4: One-hot encode categorical variables ----
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded = df_encoded.replace({True: 1, False: 0})

# ---- Step 5: Normalize numerical features ----
scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

# ---- Step 6: Train-test split ----
X = df_encoded.drop('y', axis=1)
y = df_encoded['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

Shape: (45211, 17)
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

Target distribution:
 y
no     39922
yes     5289
Name: count, dtype: int64

Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Numerical features: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Train shape: (36168, 42)
Test shape: (9043, 42)


In [33]:
# Create a folder to store CSVs
os.makedirs('bank_data', exist_ok=True)

# Combine X and y
train_df = pd.concat([y_train.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
val_df   = pd.concat([y_test.reset_index(drop=True),  X_test.reset_index(drop=True)],  axis=1)

# Save as CSVs
train_df.to_csv('bank_data/train.csv', index=False, header=True)
val_df.to_csv('bank_data/validation.csv', index=False, header=True)

print(train_df.iloc[:, 0].unique())

[0 1]


In [217]:
#   pip install xgboost

from xgboost import XGBClassifier

X_data = pd.read_csv('bank_data/train.csv')
train = X_data.drop(columns=['y'])
y_label = X_data['y']

# Define the model with hyperparameters
# 'objective' specifies the learning task and the corresponding learning objective
# For multi-class classification, use 'multi:softprob' (default for XGBClassifier with multi-class data) or 'multi:softmax'
model = XGBClassifier(objective='binary:logistic', n_estimators=100, learning_rate=0.1, max_depth=3, use_label_encoder=False, eval_metric='logloss')

# Fit the model to the training data
model.fit(train, y_label)


C:\Users\sn774\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:06:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [35]:
from sklearn.metrics import accuracy_score

data_test = pd.read_csv('bank_data/validation.csv')
test = data_test.drop(columns = ['y'])
test_label = data_test['y']


# Make predictions on the test data
y_pred = model.predict(test)

# Evaluate the model
accuracy = accuracy_score(test_label, y_pred)
print(f"Accuracy: {accuracy*100:.2f}%")


NameError: name 'model' is not defined

In [236]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Evaluation metrics
print("Accuracy:", accuracy_score(test_label, y_pred))
print("Precision:", precision_score(test_label, y_pred))
print("Recall:", recall_score(test_label, y_pred))
print("F1 Score:", f1_score(test_label, y_pred))
print("ROC AUC:", roc_auc_score(test_label, y_pred))

Accuracy: 0.9046776512219397
Precision: 0.6666666666666666
Recall: 0.3705103969754253
F1 Score: 0.47630619684082626
ROC AUC: 0.6729821865904052


In [257]:
p = model.predict(sample)

In [255]:
sample = [[30, 1, 1, 1, 1, 1, 1, 1787, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0,1,0]]


In [7]:
len(sample[0])

40

In [259]:
p = model.predict(sample)
p

array([1])

In [9]:
import tarfile

with tarfile.open('model.tar.gz') as tar:
    tar.extractall()

In [261]:
import xgboost as xgb

model = xgb.Booster()
model.load_model('xgboost-model')

In [83]:
from xgboost import XGBClassifier

# Create model instance
m = XGBClassifier()

# Load saved model
m.load_model("../model/xgboost_model.json")

,objective,'binary:logistic'
,base_score,'1.7769065E-1'
,booster,'gbtree'
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [265]:
import pandas as pd

test = pd.read_csv("data/validation.csv")


In [267]:
test['0'].value_counts()


0
0    7984
1    1058
Name: count, dtype: int64

In [269]:
test_data = test.drop(columns = ['0'])

In [271]:
test_data.shape

(9042, 42)

In [304]:
test_data[:1]

,-0.08816663982820393,-0.2372202284620528,-0.9380027029147067,0.3449643047235309,-0.2465603476137659,-0.4114531064930479,-0.25194037067217256,True,False,False.1,...,False.20,False.21,False.22,True.5,False.23,False.24,False.25,False.26,False.27,True.6
0,0.288529,-0.32327,1.705471,-0.214205,-0.24656,-0.411453,-0.25194,False,False,False,...,False,False,False,True,False,False,False,False,False,True


In [308]:
dtest = xgb.DMatrix(test_data[:10])
preds = model.predict(dtest)
preds

array([0.08661593, 0.09743977, 0.05694479, 0.01163018, 0.17577766,
       0.00501101, 0.6621327 , 0.00911968, 0.0490673 , 0.00795814],
      dtype=float32)

In [279]:
a = [1 if i >=.5 else 0 for i in preds]

In [281]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Accuracy:", accuracy_score(test['0'], a))

Accuracy: 0.8822163238221632


# Register model

In [ ]:
import mlflow

# Connect to remote MLflow server
#mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("my-first-experiment")


# Register the Model
model_name = 'XGB'
run_id= "6725bfc626ba41c3af2d2437481914b3"
model_uri = f'runs:/{run_id}/{model_name}'

with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

# Load the Model

In [11]:
import mlflow

# Connect to remote MLflow server
#mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("my-first-experiment")

<Experiment: artifact_location='mlflow-artifacts:/993918035629794039', creation_time=1774911638675, experiment_id='993918035629794039', last_update_time=1774911638675, lifecycle_stage='active', name='my-first-experiment', tags={}, workspace='default'>

In [77]:
model_name = 'XGB'
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
#y_pred = loaded_model.predict(X_test)
#y_pred[:4]

MlflowException: API request to http://127.0.0.1:5000/api/2.0/mlflow/registered-models/get failed with exception HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/2.0/mlflow/registered-models/get?name=XGB (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000021C1C7635D0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [43]:
y_pred = loaded_model.predict(test)
y_pred[:4]

array([0, 0, 0, 0])

In [58]:
# Evaluate the model
accuracy = accuracy_score(test_label, y_pred)
print(f"Accuracy: {accuracy*100:.2f}%")

Accuracy: 90.37%


# Transition the Model to Production

In [66]:
current_model_uri

'models:/XGB'

In [68]:
current_model_uri = f"models:/{model_name}/{model_version}"
production_model_name = "bank-data-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

Successfully registered model 'bank-data-prod'.
Copied version '1' of model 'XGB' to version '1' of model 'bank-data-prod'.


<ModelVersion: aliases=[], creation_timestamp=1774923822732, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1774923822732, metrics=None, model_id=None, name='bank-data-prod', params=None, run_id='6725bfc626ba41c3af2d2437481914b3', run_link='', source='models:/XGB/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [72]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(test)
y_pred[:4]

array([0, 0, 0, 0])

# ML Flow Dagshub

In [75]:
model = [loaded_model]

In [ ]:
# dagshub setup
import dagshub
dagshub.init(repo_owner='buzzworm47', repo_name='bank_demo', mlflow=True)



In [ ]:
# Ideally you will not require following 4 lines if you have started fresh and do not have any previous dagshub credentials on your computer
import os
os.environ['MLFLOW_TRACKING_USERNAME'] = 'buzzworm47' # 'learnpythonlanguage'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'your password' # 
os.environ['MLFLOW_TRACKING_URI'] = 'your dagshub unique uri' # https://dagshub.com/learnpythonlanguage/mlflow_dagshub_demo.mlflow

# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
# mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  